In [128]:
# IMPORTS
import numpy as np
import pandas as pd
import requests

#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date

# for graphs
import matplotlib.pyplot as plt

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

# display all columns in dataframe
pd.set_option('display.max_columns', None)

### Question 1: IPO Filings Web Scraping and Data Processing

**What's the total sum ($m) of 2023 filings that happened on Fridays?**

Re-use the [Code Snippet 1] example to get the data from web for this endpoint: https://stockanalysis.com/ipos/filings/
Convert the 'Filing Date' to datetime(), 'Shares Offered' to float64 (if '-' is encountered, populate with NaNs).
Define a new field 'Avg_price' based on the "Price Range", which equals to NaN if no price is specified, to the price (if only one number is provided), or to the average of 2 prices (if a range is given).
You may be inspired by the function `extract_numbers()` in [Code Snippet 4], or you can write your own function to "parse" a string.
Define a column "Shares_offered_value", which equals to "Shares Offered" * "Avg_price" (when both columns are defined; otherwise, it's NaN)

Find the total sum in $m (millions of USD, closest INTEGER number) for all filings during 2023, which happened on Fridays (`Date.dt.dayofweek()==4`). You should see 32 records in total, 25 of it is not null.

(additional: you can read about [S-1 IPO filing](https://www.dfinsolutions.com/knowledge-hub/thought-leadership/knowledge-resources/what-s-1-ipo-filing) to understand the context)

In [129]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}

url = "https://stockanalysis.com/ipos/filings/"
response = requests.get(url, headers=headers)

ipo_dfs = pd.read_html(response.text)

ipos_2023 = ipo_dfs[0]

ipos_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 326 entries, 0 to 325
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Filing Date     326 non-null    object
 1   Symbol          326 non-null    object
 2   Company Name    326 non-null    object
 3   Price Range     326 non-null    object
 4   Shares Offered  326 non-null    object
dtypes: object(5)
memory usage: 12.9+ KB


In [130]:
ipos_2023.head()

,Filing Date,Symbol,Company Name,Price Range,Shares Offered
0,"May 3, 2024",TBN,Tamboran Resources Corporation,-,-
1,"Apr 29, 2024",HWEC,"HW Electro Co., Ltd.",$3.00,3750000
2,"Apr 29, 2024",DTSQ,DT Cloud Star Acquisition Corporation,$10.00,6000000
3,"Apr 26, 2024",EURK,Eureka Acquisition Corp,$10.00,5000000
4,"Apr 26, 2024",HDL,Super Hi International Holding Ltd.,-,-


In [131]:
# convert Fillings Date to datetime
ipos_2023['Filing Date'] = pd.to_datetime(ipos_2023['Filing Date'])

# choose only fillings for 2023
ipos_2023 = ipos_2023[ipos_2023['Filing Date'].dt.year == 2023]

# replace "-" with NaN in Shares Offered and to float64
ipos_2023['Shares Offered'] = ipos_2023['Shares Offered'].replace('-', np.nan).astype(float)

# Define a new field 'Avg_price' based on the "Price Range", which equals to NaN if no price is specified, to the price
# (if only one number is provided), or to the average of 2 prices (if a range is given).
#You may be inspired by the function `extract_numbers()` in [Code Snippet 4], or you can write your own function to
# "parse" a string.

# Function to parse price range string and calculate average price
def calculate_avg_price(price_range):
    # remove $ from price
    price_range = price_range.replace('$', '')

    if price_range == '-':
        return np.nan
    elif '-' in price_range:
        prices = price_range.split('-')
        return (float(prices[0]) + float(prices[1])) / 2
    else:
        return float(price_range)

# Define a new field 'Avg_price' based on the "Price Range"
ipos_2023['Avg_price'] = ipos_2023['Price Range'].apply(calculate_avg_price)

# Define a column "Shares_offered_value", which equals to "Shares Offered" * "Avg_price" (when both columns are defined; otherwise, it's NaN)
ipos_2023['Shares_offered_value'] = ipos_2023['Shares Offered'] * ipos_2023['Avg_price']

In [132]:
ipos_2023[ipos_2023['Filing Date'].dt.dayofweek == 4].shape

(32, 7)

In [133]:
# Find the total sum in $m (millions of USD, closest INTEGER number) for all filings during 2023, which happened on Fridays (`Date.dt.dayofweek()==4`).
# You should see 32 records in total, 25 of it is not null. The total sum should be 1,000,000,000 USD.
round(ipos_2023[ipos_2023['Filing Date'].dt.dayofweek == 4]['Shares_offered_value'].sum()/1e6, 0)

286.0

In [134]:
ipos_2023.head()

,Filing Date,Symbol,Company Name,Price Range,Shares Offered,Avg_price,Shares_offered_value
50,2023-12-29,LEC,Lafayette Energy Corp,$3.50 - $4.50,1200000.0,4.0,4800000.0
51,2023-12-29,EPSM,Epsium Enterprise Limited,-,NaN,NaN,NaN
52,2023-12-28,ONDR,"Sushi Ginza Onodera, Inc.",$7.00 - $8.00,1066667.0,7.5,8000002.5
53,2023-12-27,JDZG,Jiade Limited,$4.00 - $5.00,2200000.0,4.5,9900000.0
54,2023-12-22,CHLW,Chun Hui Le Wan International Holding Group Ltd,-,NaN,NaN,NaN


### Question 2:  IPOs "Fixed days hold" strategy


**Find the optimal number of days X (between 1 and 30), where 75% quantile growth is the highest?**


Reuse [Code Snippet 1] to retrieve the list of IPOs from 2023 and 2024 (from URLs: https://stockanalysis.com/ipos/2023/ and https://stockanalysis.com/ipos/2024/).
Get all OHLCV daily prices for all stocks with an "IPO date" before March 1, 2024 ("< 2024-03-01") - 184 tickers (without 'RYZB'). Please remove 'RYZB', as it is no longer available on Yahoo Finance.

Sometimes you may need to adjust the symbol name (e.g., 'IBAC' on stockanalysis.com -> 'IBACU' on Yahoo Finance) to locate OHLCV prices for all stocks. Also, you can see the ticker changes using this [link](https://stockanalysis.com/actions/changes/).
Some of the tickers (like 'DYCQ' and 'LEGT') were on the market less than 30 days (11 and 21 days, respectively). Let's leave them in the dataset; it just means that you couldn't hold them for more days than they were listed.

Let's assume you managed to buy a new stock (listed on IPO) on the first day at the [Adj Close] price]. Your strategy is to hold for exactly X full days (where X is between 1 and 30) and sell at the "Adj. Close" price in X days (e.g., if X=1, you sell on the next day).
Find X, when the 75% quantile growth (among 185 investments) is the highest.

HINTs:
* You can generate 30 additional columns: growth_future_1d ... growth_future_30d, join that with the table of min_dates (first day when each stock has data on Yahoo Finance), and perform vector operations on the resulting dataset.
* You can use the `DataFrame.describe()` function to get mean, min, max, 25-50-75% quantiles.


Additional:
* You can also ensure that the mean and 50th percentile (median) investment returns are negative for most X values, implying a wager for a "lucky" investor who might be in the top 25%.
* What's your recommendation: Do you suggest pursuing this strategy for an optimal X?

In [135]:
# https://stockanalysis.com/ipos/2023/
# https://stockanalysis.com/ipos/2024/

# retrieve the list of IPOs form 2023 and 2024
url_2023 = "https://stockanalysis.com/ipos/2023/"
response_2023 = requests.get(url_2023, headers=headers)

ipos_2023 = pd.read_html(response_2023.text)[0]

url_2024 = "https://stockanalysis.com/ipos/2024/"
response_2024 = requests.get(url_2024, headers=headers)

ipos_2024 = pd.read_html(response_2024.text)[0]

# convert dates to datetime objects
ipos_2023['IPO Date'] = pd.to_datetime(ipos_2023['IPO Date'])
ipos_2024['IPO Date'] = pd.to_datetime(ipos_2024['IPO Date'])

## stacked IPOs
stacked_ipos_df = pd.concat([ipos_2023, ipos_2024], ignore_index=True)

stacked_ipos_df.head()

,IPO Date,Symbol,Company Name,IPO Price,Current,Return
0,2023-12-27,IROH,Iron Horse Acquisitions Corp.,$10.00,$10.04,0.40%
1,2023-12-19,LGCB,Linkage Global Inc,$4.00,$2.91,-27.50%
2,2023-12-15,ZKH,ZKH Group Limited,$15.50,$12.22,-21.16%
3,2023-12-15,BAYA,Bayview Acquisition Corp,$10.00,$10.19,1.80%
4,2023-12-14,INHD,Inno Holdings Inc.,$4.00,$0.62,-84.60%


In [136]:
# no missing prices
stacked_ipos_df[stacked_ipos_df['IPO Price'].astype(str).str.find('-') >= 0]

,IPO Date,Symbol,Company Name,IPO Price,Current,Return


In [137]:
# no missing current prices
stacked_ipos_df[stacked_ipos_df['Current'].astype(str).str.find('-') >= 0]

,IPO Date,Symbol,Company Name,IPO Price,Current,Return


In [138]:
stacked_ipos_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   IPO Date      219 non-null    datetime64[ns]
 1   Symbol        219 non-null    object        
 2   Company Name  219 non-null    object        
 3   IPO Price     219 non-null    object        
 4   Current       219 non-null    object        
 5   Return        219 non-null    object        
dtypes: datetime64[ns](1), object(5)
memory usage: 10.4+ KB


In [139]:
# convert prices to numeric
stacked_ipos_df['Current'] = pd.to_numeric(stacked_ipos_df['Current'].str.replace('$', ''), errors='coerce')
stacked_ipos_df['IPO Price'] = pd.to_numeric(stacked_ipos_df['IPO Price'].str.replace('$', ''), errors='coerce')
stacked_ipos_df['Return'] = pd.to_numeric(stacked_ipos_df['Return'].str.replace('%', ''), errors='coerce') / 100

In [140]:
# price increase
stacked_ipos_df['Price Increase'] = stacked_ipos_df['Current'] - stacked_ipos_df['IPO Price']

stacked_ipos_df.head()

,IPO Date,Symbol,Company Name,IPO Price,Current,Return,Price Increase
0,2023-12-27,IROH,Iron Horse Acquisitions Corp.,10.0,10.04,0.0040,0.04
1,2023-12-19,LGCB,Linkage Global Inc,4.0,2.91,-0.2750,-1.09
2,2023-12-15,ZKH,ZKH Group Limited,15.5,12.22,-0.2116,-3.28
3,2023-12-15,BAYA,Bayview Acquisition Corp,10.0,10.19,0.0180,0.19
4,2023-12-14,INHD,Inno Holdings Inc.,4.0,0.62,-0.8460,-3.38


In [141]:
stacked_ipos_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   IPO Date        219 non-null    datetime64[ns]
 1   Symbol          219 non-null    object        
 2   Company Name    219 non-null    object        
 3   IPO Price       219 non-null    float64       
 4   Current         219 non-null    float64       
 5   Return          217 non-null    float64       
 6   Price Increase  219 non-null    float64       
dtypes: datetime64[ns](1), float64(4), object(2)
memory usage: 12.1+ KB


In [142]:
# checking for null values
stacked_ipos_df.isnull().sum()

IPO Date          0
Symbol            0
Company Name      0
IPO Price         0
Current           0
Return            2
Price Increase    0
dtype: int64

In [143]:
stacked_ipos_df.describe()

,IPO Price,Current,Return,Price Increase
count,219.000000,219.000000,217.000000,219.000000
mean,11.027991,11.113607,-0.197956,0.085616
std,11.229966,16.985739,0.652324,8.977968
min,2.500000,0.000000,-0.999600,-20.440000
25%,4.000000,1.255000,-0.724000,-3.645000
50%,8.000000,5.730000,-0.242500,-1.550000
75%,13.500000,10.860000,0.059000,0.630000
max,92.000000,118.270000,2.672500,55.070000


In [144]:
# use only the "IPO date" before March 1, 2024 ("< 2024-03-01")
stacked_ipos_df = stacked_ipos_df[stacked_ipos_df['IPO Date'] < '2024-03-01']

In [145]:
# Truncate to the first day in the month - for Bar names
stacked_ipos_df['Date_monthly'] = stacked_ipos_df['IPO Date'].dt.to_period('M').dt.to_timestamp()

# Count the number of deals for each month and year
monthly_deals = stacked_ipos_df['Date_monthly'].value_counts().reset_index().sort_values(by='Date_monthly')
monthly_deals.columns = ['Date_monthly', 'Number of Deals']

# Plotting the bar chart using Plotly Express
fig = px.bar(monthly_deals,
             x='Date_monthly',
             y='Number of Deals',
             labels={'Month_Year': 'Month and Year', 'Number of Deals': 'Number of Deals'},
             title='Number of IPO Deals per Month and Year',
             text='Number of Deals'
             )
fig.update_traces(textposition='outside', # Position the text outside the bars
                  textfont=dict(color='black',size=14), # Adjust the font size of the text
                  )
fig.update_layout(title_x=0.5) # Center the title

fig.show()

In [146]:
#get the list of all IPOs
stacked_ipos_df['IPO Date'].dt.year.value_counts()

2023    154
2024     31
Name: IPO Date, dtype: int64

In [147]:
stacked_ipos_df['IPO Date']

0     2023-12-27
1     2023-12-19
2     2023-12-15
3     2023-12-15
4     2023-12-14
         ...    
214   2024-01-18
215   2024-01-18
216   2024-01-12
217   2024-01-11
218   2024-01-09
Name: IPO Date, Length: 185, dtype: datetime64[ns]

In [148]:
# retrieve the list of IPOs form 2023
url_2023_changes = "https://stockanalysis.com/actions/changes/2023/"
response_changes_2023 = requests.get(url_2023_changes, headers=headers)
ipos_name_changes_2023_dfs = pd.read_html(response_changes_2023.text)[0]

# retrieve the list of IPOs form 2024
url_2024_changes = "https://stockanalysis.com/actions/changes/2024/"
response_changes_2024 = requests.get(url_2024_changes, headers=headers)
ipos_name_changes_2024_dfs = pd.read_html(response_changes_2024.text)[0]

In [149]:
# removing RYZB from stacked_ipos_df
stacked_ipos_df = stacked_ipos_df[stacked_ipos_df['Symbol'] != 'RYZB']
stacked_ipos_df.head(3)

,IPO Date,Symbol,Company Name,IPO Price,Current,Return,Price Increase,Date_monthly
0,2023-12-27,IROH,Iron Horse Acquisitions Corp.,10.0,10.04,0.0040,0.04,2023-12-01
1,2023-12-19,LGCB,Linkage Global Inc,4.0,2.91,-0.2750,-1.09,2023-12-01
2,2023-12-15,ZKH,ZKH Group Limited,15.5,12.22,-0.2116,-3.28,2023-12-01


In [150]:
# get all symbols for stacked_ipos_df
symbols = stacked_ipos_df['Symbol'].unique()
symbols

array(['IROH', 'LGCB', 'ZKH', 'BAYA', 'INHD', 'AFJK', 'GSIW', 'FEBO',
       'CLBR', 'ELAB', 'RR', 'DDC', 'SHIM', 'GLAC', 'SGN', 'HG', 'CRGX',
       'ANSC', 'AITR', 'GVH', 'LXEO', 'PAPL', 'ATGL', 'MNR', 'WBUY',
       'NCL', 'BIRK', 'GMM', 'PMEC', 'LRHC', 'GPAK', 'SPKL', 'QETA',
       'MSS', 'ANL', 'SYRA', 'VSME', 'LRE', 'TURB', 'MDBH', 'KVYO',
       'CART', 'DTCK', 'NMRA', 'ARM', 'SPPL', 'NWGL', 'SWIN', 'IVP',
       'NNAG', 'SRM', 'SPGC', 'LQR', 'NRXS', 'FTEL', 'MIRA', 'PXDT',
       'HRYU', 'CTNT', 'SRFM', 'PRZO', 'HYAC', 'KVAC', 'JNVR', 'ELWS',
       'WRNT', 'TSBX', 'ODD', 'APGE', 'NETD', 'SGMT', 'BOWN', 'SXTP',
       'PWM', 'VTMX', 'INTS', 'SVV', 'KGS', 'FIHL', 'GENK', 'BUJA', 'BOF',
       'AZTR', 'CAVA', 'ESHA', 'ATMU', 'ATS', 'IPXX', 'CWD', 'SGE',
       'SLRN', 'ALCY', 'KVUE', 'GODN', 'TRNR', 'AACT', 'JYD', 'USGO',
       'UCAR', 'WLGS', 'TPET', 'TCJH', 'GDTC', 'VCIG', 'GDHG', 'ARBB',
       'ISPR', 'MGIH', 'MWG', 'HSHP', 'SFWL', 'SYT', 'HKIT', 'CHSN',
       'TBMC', 'HLP

In [151]:
list_of_symbols_that_do_not_exist = ['PTHR','DYCQ','LEGT','JVSA']

In [152]:
# what are the company names for list_of_symbols_that_do_not_exist
stacked_ipos_df[stacked_ipos_df['Symbol'].isin(list_of_symbols_that_do_not_exist)]

,IPO Date,Symbol,Company Name,IPO Price,Current,Return,Price Increase,Date_monthly
137,2023-02-10,PTHR,"Pono Capital Three, Inc.",10.0,5.72,-0.4280,-4.28,2023-02-01
190,2024-02-21,DYCQ,DT Cloud Acquisition Corporation,10.0,10.16,0.0160,0.16,2024-02-01
200,2024-02-06,LEGT,Legato Merger Corp. III,10.0,10.10,0.0100,0.10,2024-02-01
213,2024-01-19,JVSA,JVSPAC Acquisition Corp.,10.0,10.09,0.0094,0.09,2024-01-01


In [153]:
# map new values for this symbols
map_new_values = {'PTHR': 'HOVR',
                   'DYCQ': 'DYCQU',
                   'LEGT': 'LEGT-UN'}


stacked_ipos_df['Symbol'] = stacked_ipos_df['Symbol'].replace(map_new_values)

In [154]:
symbols = stacked_ipos_df['Symbol'].unique()
symbols

array(['IROH', 'LGCB', 'ZKH', 'BAYA', 'INHD', 'AFJK', 'GSIW', 'FEBO',
       'CLBR', 'ELAB', 'RR', 'DDC', 'SHIM', 'GLAC', 'SGN', 'HG', 'CRGX',
       'ANSC', 'AITR', 'GVH', 'LXEO', 'PAPL', 'ATGL', 'MNR', 'WBUY',
       'NCL', 'BIRK', 'GMM', 'PMEC', 'LRHC', 'GPAK', 'SPKL', 'QETA',
       'MSS', 'ANL', 'SYRA', 'VSME', 'LRE', 'TURB', 'MDBH', 'KVYO',
       'CART', 'DTCK', 'NMRA', 'ARM', 'SPPL', 'NWGL', 'SWIN', 'IVP',
       'NNAG', 'SRM', 'SPGC', 'LQR', 'NRXS', 'FTEL', 'MIRA', 'PXDT',
       'HRYU', 'CTNT', 'SRFM', 'PRZO', 'HYAC', 'KVAC', 'JNVR', 'ELWS',
       'WRNT', 'TSBX', 'ODD', 'APGE', 'NETD', 'SGMT', 'BOWN', 'SXTP',
       'PWM', 'VTMX', 'INTS', 'SVV', 'KGS', 'FIHL', 'GENK', 'BUJA', 'BOF',
       'AZTR', 'CAVA', 'ESHA', 'ATMU', 'ATS', 'IPXX', 'CWD', 'SGE',
       'SLRN', 'ALCY', 'KVUE', 'GODN', 'TRNR', 'AACT', 'JYD', 'USGO',
       'UCAR', 'WLGS', 'TPET', 'TCJH', 'GDTC', 'VCIG', 'GDHG', 'ARBB',
       'ISPR', 'MGIH', 'MWG', 'HSHP', 'SFWL', 'SYT', 'HKIT', 'CHSN',
       'TBMC', 'HLP

In [155]:
stacked_ipos_df[stacked_ipos_df['Symbol'].isin(list_of_symbols_that_do_not_exist)]

,IPO Date,Symbol,Company Name,IPO Price,Current,Return,Price Increase,Date_monthly
213,2024-01-19,JVSA,JVSPAC Acquisition Corp.,10.0,10.09,0.0094,0.09,2024-01-01


In [157]:
# Get all OHLCV daily prices for all stocks with an "IPO date" before March 1,
# 2024 ("< 2024-03-01") - 184 tickers (without 'RYZB').
# loop through symbols list, if you cannot find it print
# if you find it, create a dictionary with the symbol as key and the dataframe
# as value

# create a dictionary to store the dataframes
stock_data = {}

# loop through the symbols
for symbol in symbols:
    try:
        # get the data
        stock_data[symbol] = yf.download(symbol, start='2023-01-01', end='2024-03-01')

        # create a ticker columns
        stock_data[symbol]['Ticker'] = symbol
        #create a year columns
        stock_data[symbol]['Year'] = stock_data[symbol].index.year
        #create a month columns
        stock_data[symbol]['Month'] = stock_data[symbol].index.month
        #create a weekday columns
        stock_data[symbol]['Weekday'] = stock_data[symbol].index.weekday
        #create a date columns
        stock_data[symbol]['Date'] = stock_data[symbol].index.date
        # You can generate 30 additional columns: growth_future_1d ... growth_future_30d,
        # join that with the table of min_dates (first day when each stock has data on Yahoo Finance),
        # and perform vector operations on the resulting dataset.
        for i in range(1, 31):
            stock_data[symbol][f'growth_future_{i}d'] = stock_data[symbol]['Adj Close']/stock_data[symbol]['Adj Close'].shift(i)


    except Exception as e:
        # if you cannot find it, print
        print(f"Could not find {symbol}")
        # save if that symbol in a list_of_symbols_that_do_not_exist

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%*******

Could not find JVSA


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


In [158]:
stock_data.keys()

dict_keys(['IROH', 'LGCB', 'ZKH', 'BAYA', 'INHD', 'AFJK', 'GSIW', 'FEBO', 'CLBR', 'ELAB', 'RR', 'DDC', 'SHIM', 'GLAC', 'SGN', 'HG', 'CRGX', 'ANSC', 'AITR', 'GVH', 'LXEO', 'PAPL', 'ATGL', 'MNR', 'WBUY', 'NCL', 'BIRK', 'GMM', 'PMEC', 'LRHC', 'GPAK', 'SPKL', 'QETA', 'MSS', 'ANL', 'SYRA', 'VSME', 'LRE', 'TURB', 'MDBH', 'KVYO', 'CART', 'DTCK', 'NMRA', 'ARM', 'SPPL', 'NWGL', 'SWIN', 'IVP', 'NNAG', 'SRM', 'SPGC', 'LQR', 'NRXS', 'FTEL', 'MIRA', 'PXDT', 'HRYU', 'CTNT', 'SRFM', 'PRZO', 'HYAC', 'KVAC', 'JNVR', 'ELWS', 'WRNT', 'TSBX', 'ODD', 'APGE', 'NETD', 'SGMT', 'BOWN', 'SXTP', 'PWM', 'VTMX', 'INTS', 'SVV', 'KGS', 'FIHL', 'GENK', 'BUJA', 'BOF', 'AZTR', 'CAVA', 'ESHA', 'ATMU', 'ATS', 'IPXX', 'CWD', 'SGE', 'SLRN', 'ALCY', 'KVUE', 'GODN', 'TRNR', 'AACT', 'JYD', 'USGO', 'UCAR', 'WLGS', 'TPET', 'TCJH', 'GDTC', 'VCIG', 'GDHG', 'ARBB', 'ISPR', 'MGIH', 'MWG', 'HSHP', 'SFWL', 'SYT', 'HKIT', 'CHSN', 'TBMC', 'HLP', 'ZJYL', 'TMTC', 'YGFGF', 'OAKU', 'BANL', 'OMH', 'MGRX', 'FORL', 'ICG', 'IZM', 'AESI', 'AIXI

In [173]:
# remove JVSA key
stock_data.pop('JVSA')

,Open,High,Low,Close,Adj Close,Volume,Ticker
Date,,,,,,,


In [174]:
stock_data['IROH'].head()

,Open,High,Low,Close,Adj Close,Volume,Ticker,Year,Month,Weekday,Date,growth_future_1d,growth_future_2d,growth_future_3d,growth_future_4d,growth_future_5d,growth_future_6d,growth_future_7d,growth_future_8d,growth_future_9d,growth_future_10d,growth_future_11d,growth_future_12d,growth_future_13d,growth_future_14d,growth_future_15d,growth_future_16d,growth_future_17d,growth_future_18d,growth_future_19d,growth_future_20d,growth_future_21d,growth_future_22d,growth_future_23d,growth_future_24d,growth_future_25d,growth_future_26d,growth_future_27d,growth_future_28d,growth_future_29d,growth_future_30d
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2024-02-16,10.05,10.05,10.010,10.010,10.010,16700,IROH,2024,2,4,2024-02-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-02-20,10.02,10.02,10.020,10.020,10.020,5200,IROH,2024,2,1,2024-02-20,1.000999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-02-21,10.02,10.02,10.015,10.015,10.015,98600,IROH,2024,2,2,2024-02-21,0.999501,1.000500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-02-22,10.02,10.02,10.020,10.020,10.020,5600,IROH,2024,2,3,2024-02-22,1.000499,1.000000,1.000999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-02-23,10.02,10.02,10.010,10.010,10.010,14800,IROH,2024,2,4,2024-02-23,0.999002,0.999501,0.999002,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [175]:
stock_data['LGCB']

# Find growth_future_id, when the 75% quantile growth is the highest in stock_data['IROH']
# (among all 30 columns 'growth_future_id').

# create a list to store the results
results = []

# loop through the columns
for i in range(1, 31):
    # calculate the 75% quantile
    quantile_75 = stock_data['IROH'][f'growth_future_{i}d'].quantile(0.75)
    # append the results
    results.append((i, quantile_75))

# find the maximum value
max_value = max(results, key=lambda x: x[1])

max_value


(1, 1.0006242028602326)

In [176]:
all_max = []

for stock in stock_data.keys():
    print(stock)

    results = []

    for i in range(1,31):
        quantile_75 = stock_data[stock][f'growth_future_{i}d'].quantile(0.75)

        results.append((i, quantile_75))

    # find the maximum value
    max_value = max(results, key=lambda x: x[1])

    all_max.append((stock, max_value))

    # clear the results
    results.clear()


IROH
LGCB
ZKH
BAYA
INHD
AFJK
GSIW
FEBO
CLBR
ELAB
RR
DDC
SHIM
GLAC
SGN
HG
CRGX
ANSC
AITR
GVH
LXEO
PAPL
ATGL
MNR
WBUY
NCL
BIRK
GMM
PMEC
LRHC
GPAK
SPKL
QETA
MSS
ANL
SYRA
VSME
LRE
TURB
MDBH
KVYO
CART
DTCK
NMRA
ARM
SPPL
NWGL
SWIN
IVP
NNAG
SRM
SPGC
LQR
NRXS
FTEL
MIRA
PXDT
HRYU
CTNT
SRFM
PRZO
HYAC
KVAC
JNVR
ELWS
WRNT
TSBX
ODD
APGE
NETD
SGMT
BOWN
SXTP
PWM
VTMX
INTS
SVV
KGS
FIHL
GENK
BUJA
BOF
AZTR
CAVA
ESHA
ATMU
ATS
IPXX
CWD
SGE
SLRN
ALCY
KVUE
GODN
TRNR
AACT
JYD
USGO
UCAR
WLGS
TPET
TCJH
GDTC
VCIG
GDHG
ARBB
ISPR
MGIH
MWG
HSHP
SFWL
SYT
HKIT
CHSN
TBMC
HLP
ZJYL
TMTC
YGFGF
OAKU
BANL
OMH
MGRX
FORL
ICG
IZM
AESI
AIXI
SBXC
BMR
DIST
GXAI
MARX
BFRG
ENLT
MLYS
HOVR
BLAC
NXT
HSAI
LSDI
LICN
GPCR
ASST
CETU
TXO
BREA
GNLX
QSG
CVKD
SKWD
ISRL
MGOL
SMXT
VHAI
DYCQU
CHRO
UMAC
TBBB
MGX
HLXB
TELO
KYTX
PMNT
AHR
LEGT-UN
ANRO
GUTS
AS
FBLG
BTSG
AVBP
HAO
CGON
YIBO
SUGP
JL
KSPI
PSBD
CCTG
SYNX
SDHC
ROMA


In [189]:
# Find X, when the 75% quantile growth (among 185 investments) is the highest.
all_max_df = pd.DataFrame(all_max, columns=['Stock', 'Max_75_quantile'])

#unpack the tupple in Max_75_quantile
all_max_df['Day'] = all_max_df['Max_75_quantile'].apply(lambda x: x[0])
all_max_df['Max_75_quantile'] = all_max_df['Max_75_quantile'].apply(lambda x: x[1])

all_max_df.head(2)

,Stock,Max_75_quantile,Day
0,IROH,1.000624,1
1,LGCB,1.268177,22


In [194]:
#most common days for all stocks
all_max_df['Day'].value_counts()

30    51
29    12
1      8
13     8
10     7
23     7
28     7
17     6
24     6
3      5
27     5
8      5
26     5
22     5
4      5
11     4
20     4
12     4
6      4
19     3
2      3
14     3
7      3
5      3
18     2
25     2
21     2
15     2
16     1
9      1
Name: Day, dtype: int64

### Question 3: Is Growth Concentrated in the Largest Stocks?

Get the share of days (percentage as int) when Large Stocks outperform (growth_7d - growth over 7 periods back) the Largest stocks?

Reuse [Code Snippet 5] to obtain OHLCV stats for 33 stocks for 10 full years of data (2014-01-01 to 2023-12-31). You'll need to download slightly more data (7 periods before 2014-01-01 to calculate the growth_7d for the first 6 days correctly):

US_STOCKS = ['MSFT', 'AAPL', 'GOOG', 'NVDA', 'AMZN', 'META', 'BRK-B', 'LLY', 'AVGO','V', 'JPM']

EU_STOCKS = ['NVO','MC.PA', 'ASML', 'RMS.PA', 'OR.PA', 'SAP', 'ACN', 'TTE', 'SIE.DE','IDEXY','CDI.PA']

INDIA_STOCKS = ['RELIANCE.NS','TCS.NS','HDB','BHARTIARTL.NS','IBN','SBIN.NS','LICI.NS','INFY','ITC.NS','HINDUNILVR.NS','LT.NS']

LARGEST_STOCKS = US_STOCKS + EU_STOCKS + INDIA_STOCKS

Now let's add the top 12-22 stocks (as of end-April 2024):

NEW_US = ['TSLA','WMT','XOM','UNH','MA','PG','JNJ','MRK','HD','COST','ORCL']

NEW_EU = ['PRX.AS','CDI.PA','AIR.PA','SU.PA','ETN','SNY','BUD','DTE.DE','ALV.DE','MDT','AI.PA','EL.PA']

NEW_INDIA = ['BAJFINANCE.NS','MARUTI.NS','HCLTECH.NS','TATAMOTORS.NS','SUNPHARMA.NS','ONGC.NS','ADANIENT.NS','ADANIENT.NS','NTPC.NS','KOTAKBANK.NS','TITAN.NS']

LARGE_STOCKS = NEW_EU + NEW_US + NEW_INDIA

You should be able to obtain stats for 33 LARGEST STOCKS and 32 LARGE STOCKS (from the actual stats on Yahoo Finance)

Calculate growth_7d for every stock and every day. Get the average daily growth_7d for the LARGEST_STOCKS group vs. the LARGE_STOCKS group.

For example, for the first of data you should have:
Date 	ticker_category 	growth_7d
2014-01-01 	LARGE 	1.011684
2014-01-01 	LARGEST 	1.011797

On that day, the LARGEST group was growing faster than LARGE one (new stocks).

Calculate the number of days when the LARGE GROUP (new smaller stocks) outperforms the LARGEST GROUP, divide it by the total number of trading days (which should be 2595 days), and convert it to a percentage (closest INTEGER value). For example, if you find that 1700 out of 2595 days meet this condition, it means that 1700/2595 = 0.655, or approximately 66% of days, the LARGE stocks were growing faster than the LARGEST ones. This suggests that you should consider extending your dataset with more stocks to seek higher growth.

HINT: you can use pandas.pivot_table() to "flatten" the table (LARGE and LARGEST growth_7d as columns)